# Embed unified Swiss legal corpus with Qwen3-Embedding-8B

**Target hardware:** Colab G4 / NVIDIA RTX PRO 6000 Blackwell (95 GB VRAM, sm_100).

**Input:** `unified_embedding_input.parquet` produced locally by `scripts/prepare_embedding_input.py`.
It contains `doc_id`, `family`, `citation`, `vector_text`, `char_len` for ~2.65M docs (175k law + 2.48M court).

**Output:** chunked fp16 numpy arrays plus a doc_id ordering manifest, written back to Drive:

```
qwen3_8b_unified_chunk000.npy ... qwen3_8b_unified_chunkNNN.npy   # (rows, 4096) float16, L2-normalized
qwen3_8b_unified_manifest.parquet                                  # doc_id ordering for ANN index alignment
qwen3_8b_unified_summary.json                                      # encode timing, dtype, attention backend
```

**Optimizations for Blackwell:**
- `bf16` weights (Blackwell tensor cores; more stable than fp16 for transformer activations).
- **Hardware-aware Flash-Attention backend chain.** FlashAttention-3 is Hopper-only (H100/H800) and does not run on Blackwell, so on sm_100 we skip it and use:
  1. **FlashAttention-4** if `flash-attn-4` is installed and HF transformers exposes it (via `attn_implementation`). FA4 is the CuTeDSL kernel optimised for Hopper *and* Blackwell; it ships as a pure-Python JIT wheel (`pip install flash-attn-4`) so no nvcc compile is required.
  2. **FlashAttention-2** if a Blackwell-compatible `flash-attn` wheel is available.
  3. **PyTorch SDPA** (always available). On Blackwell with PyTorch ≥ 2.7, SDPA picks the cuDNN flash-attention backend automatically — this is the actual workhorse path when neither FA2 nor FA4 are wired through HF transformers yet.
- `padding_side='left'` (Qwen3-Embedding uses last-token pooling).
- Length-sorted batching within each 100k-row chunk → minimal padding waste.
- Big batch (default 512). 95 GB VRAM with ~16 GB bf16 model + activations leaves headroom.
- `torch.compile(mode='reduce-overhead')` on the encoder backbone, with graceful fallback.
- Output saved as **fp16** to halve disk footprint (~22 GB for full corpus vs 43 GB at fp32). Dot product on L2-normalized fp16 vectors is fine for ANN retrieval.
- Resumable: existing chunk files are skipped on rerun.

**Document side has no instruction prefix.** Per Qwen3-Embedding docs, only queries get the `Instruct: ... \nQuery: ...` template — that happens at retrieval time, not in this notebook.

**References:** [Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention) · [FlashAttention-3 blog](https://pytorch.org/blog/flashattention-3/) · [`flash-attn-4` on PyPI](https://pypi.org/project/flash-attn-4/)

In [2]:
!pip -q install flash-attn --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
ERROR: Operation cancelled by user

In [1]:
# 1. Install dependencies + Flash-Attention 4 only.
#
# On Blackwell (sm_100) the practical chain is FA4 -> SDPA. We skip FA2 because:
#   - PyPI ships no prebuilt sm_100 wheel for `flash-attn` 2.x
#   - From-source builds take 30 min to several hours on Colab
#   - Official FA2 build targets don't include sm_100 anyway
#   - PyTorch SDPA on Blackwell already uses cuDNN's flash-attention backend
#
# flash-attn-4 itself is a pure-Python JIT wheel (CuTeDSL) so it installs in seconds.

%pip -q install --upgrade pip
%pip -q install "sentence-transformers>=3.3" "transformers>=4.51" "accelerate>=0.34" "pyarrow>=15" einops

# --- FlashAttention-4 (Blackwell-ready). Pick the right extra based on CUDA major version.
import subprocess, sys
try:
    import torch as _torch_probe
    _cuda_major = int((_torch_probe.version.cuda or '0.0').split('.')[0])
except Exception:
    _cuda_major = 12
_fa4_spec = '"flash-attn-4[cu13]"' if _cuda_major >= 13 else '"flash-attn-4"'
print(f'installing flash-attn-4 (cuda major detected: {_cuda_major}, spec: {_fa4_spec}) ...')
_rc = subprocess.call(f'{sys.executable} -m pip install -q {_fa4_spec}', shell=True)
if _rc != 0:
    subprocess.call(f'{sys.executable} -m pip install -q --pre {_fa4_spec}', shell=True)

# --- Quick post-install probe.
try:
    from flash_attn.cute import flash_attn_func as _fa4_probe  # noqa: F401
    print('flash-attn-4   : OK (flash_attn.cute available)')
except Exception as e:
    print(f'flash-attn-4   : NOT available ({type(e).__name__}: {e})')
print('flash-attn (FA2): skipped (not needed on Blackwell — SDPA cuDNN flash is used)')

installing flash-attn-4 (cuda major detected: 12, spec: "flash-attn-4") ...
flash-attn-4   : OK (flash_attn.cute available)
flash-attn (FA2): skipped (not needed on Blackwell — SDPA cuDNN flash is used)


In [2]:
# 4. Mount Drive (skip if running from a path that's already on local disk).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [1]:
# 2. GPU + library check. Confirm Blackwell (sm_100) and bf16 support before loading the 8B model.
import os, sys, json, math, time, gc, glob, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch

import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

assert torch.cuda.is_available(), 'GPU required'
p = torch.cuda.get_device_properties(0)
print(f'GPU            : {p.name}')
print(f'VRAM           : {p.total_memory/1e9:.1f} GB')
print(f'Compute cap.   : sm_{p.major}{p.minor}')
print(f'Torch          : {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'bf16 supported : {torch.cuda.is_bf16_supported()}')
print(f'TF32 matmul    : {torch.backends.cuda.matmul.allow_tf32}')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU            : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM           : 102.0 GB
Compute cap.   : sm_120
Torch          : 2.10.0+cu128  CUDA: 12.8
bf16 supported : True
TF32 matmul    : False


In [2]:
# 3. Configuration. Edit DRIVE_INPUT / DRIVE_OUT_DIR before running.
MODEL_NAME      = 'Qwen/Qwen3-Embedding-8B'
MODEL_DIM       = 4096
MAX_SEQ_LEN     = 768       # vector_text is short (~70-300 tokens). 768 covers >99% with no truncation cost.
BATCH_SIZE      = 256       # 95 GB VRAM @ bf16: confirmed safe. Drop to 256 if OOM after compile.
CHUNK_SIZE      = 100_000   # rows per .npy file
OUTPUT_DTYPE    = np.float16
USE_TORCH_COMPILE = False    # set False if compile crashes on Blackwell with this torch build

DRIVE_INPUT     = '/content/drive/MyDrive/swiss_law/data/unified_embedding_input.parquet'
DRIVE_OUT_DIR   = '/content/drive/MyDrive/swiss_law/artifacts/embeddings'
OUT_PREFIX      = 'qwen3_8b_unified'

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
print(f'Input  : {DRIVE_INPUT}')
print(f'Output : {DRIVE_OUT_DIR}/{OUT_PREFIX}_chunk*.npy')

Input  : /content/drive/MyDrive/swiss_law/data/unified_embedding_input.parquet
Output : /content/drive/MyDrive/swiss_law/artifacts/embeddings/qwen3_8b_unified_chunk*.npy


In [3]:
# 5. Load parquet. Keep it in pandas; 2.65M rows × ~5 columns ≈ 2-3 GB RAM.
t0 = time.time()
df = pd.read_parquet(DRIVE_INPUT)
print(f'rows           : {len(df):,}')
print(f'load time      : {time.time()-t0:.1f}s')
print(f'columns        : {list(df.columns)}')
print(f'family counts  : {df["family"].value_counts().to_dict()}')
print(f'avg chars      : {df["char_len"].mean():.0f}')
print(f'p50/p90/p99    : {df["char_len"].quantile([0.5,0.9,0.99]).round().to_dict()}')
print(f'max chars      : {df["char_len"].max():,}')

rows           : 2,652,248
load time      : 9.7s
columns        : ['doc_id', 'family', 'citation', 'vector_text', 'char_len']
family counts  : {'court': 2476315, 'law': 175933}
avg chars      : 373
p50/p90/p99    : {0.5: 309.0, 0.9: 612.0, 0.99: 1414.0}
max chars      : 2,845


In [4]:
# 6. Load Qwen3-Embedding-8B with the best attention backend available for this GPU.
#
# Hardware-aware chain:
#   Blackwell (sm_100+):  flash_attention_4* -> flash_attention_2 -> sdpa
#   Hopper    (sm_90):    flash_attention_3  -> flash_attention_2 -> sdpa
#   Older     (sm_8x):    flash_attention_2  -> sdpa
#
# *flash_attention_4 is exposed by HF transformers only when the runtime detects flash-attn-4
#  AND the transformers version has wired it through `attn_implementation`. As of late 2025
#  this integration is in progress (see huggingface/transformers#42405). When unavailable,
#  we fall through to FA2 / SDPA. SDPA on Blackwell + torch>=2.7 already uses cuDNN's flash
#  backend internally, so the SDPA fallback is fast — not a wooden-leg path.

from sentence_transformers import SentenceTransformer
from transformers import AutoConfig

_p = torch.cuda.get_device_properties(0)
_cc = _p.major * 10 + _p.minor   # 80=A100, 89=Ada, 90=Hopper, 100/120=Blackwell
_is_blackwell = _cc >= 100
_is_hopper    = _cc == 90

# Per-arch preference list (most preferred first).
if _is_blackwell:
    _candidates = ('flash_attention_4', 'flash_attention_2', 'sdpa')
elif _is_hopper:
    _candidates = ('flash_attention_3', 'flash_attention_2', 'sdpa')
else:
    _candidates = ('flash_attention_2', 'sdpa')

# Probe what's actually importable.
_have_fa4 = False
try:
    from flash_attn.cute import flash_attn_func as _probe_fa4  # noqa: F401
    _have_fa4 = True
except Exception:
    pass
_have_fa2 = False
try:
    # Must check for the actual symbol — flash-attn-4 owns the `flash_attn`
    # namespace too, so a bare `import flash_attn` is NOT enough to confirm FA2.
    from flash_attn import flash_attn_func as _probe_fa2_func  # noqa: F401
    from flash_attn.bert_padding import unpad_input as _probe_fa2_unpad  # noqa: F401
    _have_fa2 = True
except Exception:
    pass

except Exception:
    pass
_have_fa3 = False
try:
    from transformers.modeling_flash_attention_utils import is_flash_attn_3_available
    _have_fa3 = bool(is_flash_attn_3_available())
except Exception:
    _have_fa3 = False

# Discover which strings the installed transformers version actually accepts.
try:
    from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS as _ATTN_REGISTRY
    _registry_keys = set(_ATTN_REGISTRY.valid_keys()) if hasattr(_ATTN_REGISTRY, 'valid_keys') \
                     else set(_ATTN_REGISTRY.keys())
except Exception:
    _registry_keys = {'sdpa', 'eager', 'flash_attention_2'}

def _candidate_ok(name: str) -> bool:
    if name == 'flash_attention_4':
        return _have_fa4 and ('flash_attention_4' in _registry_keys)
    if name == 'flash_attention_3':
        return _have_fa3 and ('flash_attention_3' in _registry_keys)
    if name == 'flash_attention_2':
        return _have_fa2 and ('flash_attention_2' in _registry_keys)
    return name in _registry_keys  # sdpa / eager always present

ATTN_IMPL = next((c for c in _candidates if _candidate_ok(c)), 'sdpa')

print(f'compute cap.   : sm_{_cc}  (blackwell={_is_blackwell}, hopper={_is_hopper})')
print(f'flash-attn-4   : {"available" if _have_fa4 else "missing"}')
print(f'flash-attn (FA2): {"available" if _have_fa2 else "missing"}')
print(f'flash-attn-3   : {"available" if _have_fa3 else "missing"}')
print(f'attention      : {ATTN_IMPL}')

if ATTN_IMPL == 'sdpa':
    # Hint PyTorch toward the cuDNN flash-attention backend on Blackwell / Hopper.
    # This is the path that gives "FlashAttention performance" via PyTorch's built-in
    # dispatcher when no FA package is wired into transformers.
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_cudnn_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    print('sdpa kernels   : flash + cudnn + mem-efficient enabled')

t0 = time.time()
model = SentenceTransformer(
    MODEL_NAME, device='cuda',
    model_kwargs={'torch_dtype': torch.bfloat16, 'attn_implementation': ATTN_IMPL},
    tokenizer_kwargs={'padding_side': 'left'},
)
model.max_seq_length = MAX_SEQ_LEN
model.eval()
print(f'load time      : {time.time()-t0:.1f}s')

if USE_TORCH_COMPILE:
    try:
        model[0].auto_model = torch.compile(
            model[0].auto_model, mode='reduce-overhead', fullgraph=False,
        )
        print('torch.compile  : on (reduce-overhead)')
    except Exception as exc:
        print(f'torch.compile  : skipped ({exc})')

print(f'allocated VRAM : {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'reserved VRAM  : {torch.cuda.memory_reserved()/1e9:.2f} GB')

compute cap.   : sm_120  (blackwell=True, hopper=False)
flash-attn-4   : available
flash-attn (FA2): missing
flash-attn-3   : missing
attention      : sdpa
sdpa kernels   : flash + cudnn + mem-efficient enabled


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

load time      : 7.6s
allocated VRAM : 15.13 GB
reserved VRAM  : 15.14 GB


In [5]:
# 7. Chunked encoder with length-sorted batching and fp16 save. Resumable across runs.
def encode_chunk(model, texts: list[str], batch_size: int) -> np.ndarray:
    """Encode a chunk with length-sorted batching, restore original order, return fp32 normalized embeddings."""
    n = len(texts)
    order = sorted(range(n), key=lambda i: len(texts[i]))
    sub_sorted = [texts[i] for i in order]
    with torch.inference_mode():
        emb_sorted = model.encode(
            sub_sorted,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    inv = np.empty(n, dtype=np.int64)
    for new_i, orig_i in enumerate(order):
        inv[orig_i] = new_i
    return emb_sorted[inv]


def encode_corpus(df, *, out_dir: str, prefix: str, chunk_size: int, batch_size: int):
    n = len(df)
    n_chunks = math.ceil(n / chunk_size)
    timings = []
    print(f'corpus rows    : {n:,}')
    print(f'chunks         : {n_chunks}  (chunk_size={chunk_size}, batch={batch_size})')

    for c in range(n_chunks):
        path = os.path.join(out_dir, f'{prefix}_chunk{c:03d}.npy')
        s, e = c * chunk_size, min((c + 1) * chunk_size, n)
        if os.path.exists(path):
            arr = np.load(path, mmap_mode='r')
            assert arr.shape[0] == (e - s), f'cached chunk {c} has wrong shape: {arr.shape}'
            print(f'  [cached] chunk {c+1}/{n_chunks}  rows={e-s:,}  shape={arr.shape}')
            continue

        texts = df['vector_text'].iloc[s:e].tolist()
        print(f'  encoding chunk {c+1}/{n_chunks}  rows={len(texts):,}  rng={s:,}..{e:,}', flush=True)
        t0 = time.time()
        emb = encode_chunk(model, texts, batch_size).astype(OUTPUT_DTYPE)
        dt = time.time() - t0
        timings.append(dt)
        np.save(path, emb)
        rate = len(texts) / dt
        gb = emb.nbytes / 1e9
        print(f'    saved={path}  shape={emb.shape}  dtype={emb.dtype}  '
              f'time={dt:.1f}s  rate={rate:.0f}/s  size={gb:.2f}GB', flush=True)
        del emb; gc.collect(); torch.cuda.empty_cache()

    return timings


print(f'encoding {len(df):,} passages with {MODEL_NAME} @ bf16 / {ATTN_IMPL}')

encoding 2,652,248 passages with Qwen/Qwen3-Embedding-8B @ bf16 / sdpa


In [6]:
# 8. Run the full encoding job. Watch chunk timings; the first chunk is slower because torch.compile traces graphs on first run.
timings = encode_corpus(
    df,
    out_dir=DRIVE_OUT_DIR,
    prefix=OUT_PREFIX,
    chunk_size=CHUNK_SIZE,
    batch_size=BATCH_SIZE,
)
if timings:
    print(f'\nfinished {len(timings)} chunk(s) this run.')
    print(f'  total      : {sum(timings)/60:.1f} min')
    print(f'  avg/chunk  : {sum(timings)/len(timings):.1f} s')
    print(f'  first      : {timings[0]:.1f} s  (includes compile/trace)')
    if len(timings) > 1:
        steady = sum(timings[1:]) / len(timings[1:])
        print(f'  steady-state avg: {steady:.1f} s')

corpus rows    : 2,652,248
chunks         : 27  (chunk_size=100000, batch=256)
  [cached] chunk 1/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 2/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 3/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 4/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 5/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 6/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 7/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 8/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 9/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 10/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 11/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 12/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 13/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 14/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 15/27  rows=100,000  shape=(100000, 4096)
  [cached] chunk 16/27  rows=100,000  shape=(

In [7]:
# 9. Write the doc_id manifest (ordering matches concat of chunk000..N) and a summary.
manifest_path = os.path.join(DRIVE_OUT_DIR, f'{OUT_PREFIX}_manifest.parquet')
manifest = df[['doc_id', 'family', 'citation']].copy()
manifest['row_index'] = np.arange(len(manifest), dtype=np.int64)
manifest.to_parquet(manifest_path, index=False)
print(f'manifest       : {manifest_path}  ({len(manifest):,} rows)')

summary = {
    'model_name': MODEL_NAME,
    'embedding_dim': MODEL_DIM,
    'output_dtype': str(np.dtype(OUTPUT_DTYPE)),
    'normalized': True,
    'document_prompt_prefix': None,
    'attention_impl': ATTN_IMPL,
    'max_seq_length': MAX_SEQ_LEN,
    'batch_size': BATCH_SIZE,
    'chunk_size': CHUNK_SIZE,
    'torch_compile': USE_TORCH_COMPILE,
    'chunk_count': math.ceil(len(df) / CHUNK_SIZE),
    'corpus_rows': int(len(df)),
    'family_counts': df['family'].value_counts().to_dict(),
    'gpu': torch.cuda.get_device_name(0),
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
}
summary_path = os.path.join(DRIVE_OUT_DIR, f'{OUT_PREFIX}_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'summary        : {summary_path}')
print(json.dumps(summary, indent=2))

manifest       : /content/drive/MyDrive/swiss_law/artifacts/embeddings/qwen3_8b_unified_manifest.parquet  (2,652,248 rows)
summary        : /content/drive/MyDrive/swiss_law/artifacts/embeddings/qwen3_8b_unified_summary.json
{
  "model_name": "Qwen/Qwen3-Embedding-8B",
  "embedding_dim": 4096,
  "output_dtype": "float16",
  "normalized": true,
  "document_prompt_prefix": null,
  "attention_impl": "sdpa",
  "max_seq_length": 768,
  "batch_size": 256,
  "chunk_size": 100000,
  "torch_compile": false,
  "chunk_count": 27,
  "corpus_rows": 2652248,
  "family_counts": {
    "court": 2476315,
    "law": 175933
  },
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "torch_version": "2.10.0+cu128",
  "cuda_version": "12.8"
}


In [8]:
# 10. Sanity check the saved chunks: shapes, dtypes, L2 norms ≈ 1, no NaNs.
chunk_paths = sorted(glob.glob(os.path.join(DRIVE_OUT_DIR, f'{OUT_PREFIX}_chunk*.npy')))
print(f'found {len(chunk_paths)} chunk file(s)')
total_rows = 0
for path in chunk_paths:
    arr = np.load(path, mmap_mode='r')
    total_rows += arr.shape[0]
    norms = np.linalg.norm(arr[: min(2000, len(arr))].astype(np.float32), axis=1)
    nan_count = int(np.isnan(arr[: min(2000, len(arr))]).sum())
    print(f'  {os.path.basename(path):40s} '
          f'shape={arr.shape} dtype={arr.dtype} '
          f'norm_mean={norms.mean():.4f} norm_std={norms.std():.4f} nans={nan_count}')
print(f'\ntotal rows across chunks: {total_rows:,}')
assert total_rows == len(df), f'row count mismatch: {total_rows} vs {len(df)}'

found 27 chunk file(s)
  qwen3_8b_unified_chunk000.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0018 nans=0
  qwen3_8b_unified_chunk001.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0017 nans=0
  qwen3_8b_unified_chunk002.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0018 nans=0
  qwen3_8b_unified_chunk003.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0018 nans=0
  qwen3_8b_unified_chunk004.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0017 nans=0
  qwen3_8b_unified_chunk005.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0017 nans=0
  qwen3_8b_unified_chunk006.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0017 nans=0
  qwen3_8b_unified_chunk007.npy            shape=(100000, 4096) dtype=float16 norm_mean=1.0013 norm_std=0.0017 nans=0
  qwen3_8b_unified_chunk008.npy  

In [9]:
# 11. Optional: quick query/document encode comparison — sanity that the model works for retrieval.
# Encodes a synthetic English query through the Qwen3-Embedding instruction template and looks up
# the top match in chunk 0. This is only a smoke test; real retrieval happens in a separate notebook
# against the merged embeddings + the SQLite filters.
QWEN_INSTRUCT = (
    'Instruct: Given an English-language legal question or scenario about Swiss federal law, '
    'retrieve the Swiss statute articles or federal court decision considerations that are most '
    'directly relevant to answering it.\nQuery: '
)
queries = [
    'When can Swiss courts extend pretrial detention based on collusion risk?',
    'What principles bind persons performing public tasks under Swiss public law?',
    'Dublin transfer detention proportionality test',
]
with torch.inference_mode():
    q_emb = model.encode(
        [QWEN_INSTRUCT + q for q in queries],
        batch_size=8, normalize_embeddings=True, convert_to_numpy=True,
    ).astype(np.float32)

doc_emb = np.load(chunk_paths[0], mmap_mode='r').astype(np.float32)
scores = q_emb @ doc_emb.T  # (n_queries, chunk_rows)
topk = np.argsort(-scores, axis=1)[:, :5]
for qi, q in enumerate(queries):
    print(f'\nQ: {q}')
    for rank, ri in enumerate(topk[qi]):
        row = df.iloc[ri]
        print(f'  {rank+1}. [{row.family}] {row.citation:38s}  score={scores[qi,ri]:.3f}')


Q: When can Swiss courts extend pretrial detention based on collusion risk?
  1. [court] 1B_227/2017 E. C                        score=0.780
  2. [court] 1B_541/2020 E. 3.2                      score=0.771
  3. [court] 1B_246/2007 20.11.2007 E. 2             score=0.758
  4. [court] 1P.482/2006 17.08.2006 E. B             score=0.746
  5. [court] 1B_228/2008 02.09.2008 E. A             score=0.745

Q: What principles bind persons performing public tasks under Swiss public law?
  1. [court] 2C_1023/2021 E. 2.2.1                   score=0.675
  2. [court] BGE 144 II 281 E. 4.1                   score=0.663
  3. [court] 2C_94/2018 E. 4.1                       score=0.662
  4. [court] BGE 137 II 409 E. 7.3.1                 score=0.641
  5. [court] 2C_492/2022 E. 7                        score=0.640

Q: Dublin transfer detention proportionality test
  1. [court] 2C_199/2018 E. 4.2                      score=0.735
  2. [court] 2C_142/2023 E. 3.3.5                    score=0.672
  3. [court